# NCM Concept-Space DPO & GRPO/RLVR Walkthrough

Minimal clean-room Stage-1 Residual VQ-VAE, Stage-2 Next-Concept Transformer, Concept-GRPO, and Concept-DPO workflow.

In [ ]:
import torch
from ncm_train import (
    Stage1Config, Stage2Config, GRPOConfig, DPOConfig,
    train_stage1_vqvae, train_stage2_ncm, train_stage2_grpo, train_stage2_dpo
)
from ncm_bench.frontier import generate_synthetic_dpo_preferences

vocab, seq_len = 16, 16
seqs = torch.randint(0, vocab, (24, seq_len))
pfxs = torch.randint(0, vocab, (24, seq_len))
s1_cfg = Stage1Config(in_vocab_size=vocab, context_length=seq_len, embed_dim=32, levels=(2,4,8,16), vq_vocab_size=16, n_layers=1, steps=4)
vqvae = train_stage1_vqvae(seqs, s1_cfg)['model']
s2_cfg = Stage2Config(prefix_len=seq_len, n_layer=1, n_head=2, n_embd=32, steps=4, query_level0=True)
policy = train_stage2_ncm(pfxs, seqs, vqvae, s1_cfg, s2_cfg)['model']
reward_fn = lambda logits, _p: (logits.argmax(-1) < vocab // 2).float().mean(-1)
prefs = generate_synthetic_dpo_preferences(pfxs, vqvae, policy, reward_fn, top_k=4)
dpo_out = train_stage2_dpo(pfxs, prefs['chosen_codes'], prefs['rejected_codes'], vqvae, policy, DPOConfig(steps=4))
print('Final DPO loss:', dpo_out['final_loss'], 'Accuracy:', dpo_out['preference_accuracy'])